# Lab 5 — Give It Hands
**Session 5 · Tool use + choosing the approach · TCE**

Today: fix Session 1's broken math with a real calculator, chain tools, then decide when tools are even the right answer. **File → Save a copy in Drive.**

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai
from getpass import getpass
from google import genai
from google.genai import types
import time, re

MOCK = False   # ← set True only if the instructor says the live API is unavailable

client = genai.Client(api_key="mock" if MOCK else getpass("Gemini API key: "))
MODEL = "gemini-flash-lite-latest"  # lite: minimal thinking by default (near-zero thinking tokens on simple prompts), so the free tier goes further (Aug 2026 → Gemini 3.5 Flash Lite). 429 = rate limit: the helper below waits and retries; 503 = high demand: wait and re-run.

# ---- the offline path: a stand-in for client.models.generate_content --------------------
# Manual mode (automatic calling disabled) → it emits a FunctionCall for any prompt with digits,
# exactly the shape the real model returns. Automatic mode → it runs your tool itself and answers.
_MOCK_ANSWERS = [   # (keyword, canned reply) for prompts that need no tool
    ("thirukkural", "The Thirukkural was written by Thiruvalluvar."),
    ("weather",     "I don't have live weather data. Carry an umbrella in Madurai between October and December."),
    ("without",     "18% of 2347 is about 422, so the total is roughly 2769 rupees."),   # a no-tool guess — check the digits
]
_MOCK_DEFAULT = "I can answer that directly, no tool needed."

def _mock_next_call(contents, tools):
    """What would the model ask for next? Reads the prompt + which tool results are already in the conversation."""
    items = contents if isinstance(contents, list) else [contents]
    text = " ".join(c for c in items if isinstance(c, str))
    done = [pt.function_response.name for c in items if isinstance(c, types.Content)
            for pt in (c.parts or []) if pt.function_response]
    names = {t.__name__ for t in (tools or []) if callable(t)}
    dates = re.findall(r"\d{4}-\d{2}-\d{2}", text)
    if "search_notes" in names and "notes" in text.lower() and "search_notes" not in done:
        return types.FunctionCall(name="search_notes", args={"query": text[:80]})
    if "days_between" in names and len(dates) >= 2 and "days_between" not in done:
        return types.FunctionCall(name="days_between", args={"date1": dates[0], "date2": dates[1]})
    if "calculator" in names and re.search(r"\d", text) and "calculator" not in done:
        m = re.search(r"(\d+(?:\.\d+)?)\s*%\s*(?:gst\s*)?(?:on|of)\s*(?:a bill of\s*)?(\d+(?:\.\d+)?)", text, re.I)
        nums = re.findall(r"\d+(?:\.\d+)?", re.sub(r"\d{4}-\d{2}-\d{2}", "", text))   # dates are not numbers
        expr = f"{m.group(2)} * {float(m.group(1)) / 100}" if m else " * ".join(nums[:2]) if len(nums) > 1 else (nums[0] if nums else "0")
        plus = re.search(r"plus\s+(\d+)", text, re.I)
        return types.FunctionCall(name="calculator", args={"expression": expr + (f" + {plus.group(1)}" if plus else "")})
    return None

def _mock_response(parts):
    return types.GenerateContentResponse(candidates=[types.Candidate(content=types.Content(role="model", parts=parts))])

def mock_generate(model=None, contents=None, config=None):
    tools = list(getattr(config, "tools", None) or [])
    manual = bool(getattr(getattr(config, "automatic_function_calling", None), "disable", False))
    items = list(contents) if isinstance(contents, list) else [contents]
    fc = _mock_next_call(items, tools)
    if manual:
        if fc: return _mock_response([types.Part(function_call=fc)])
        done = [f"{pt.function_response.name} → {pt.function_response.response.get('result')}"
                for c in items if isinstance(c, types.Content) for pt in (c.parts or []) if pt.function_response]
        text = "Using the tool results: " + "; ".join(done) if done else next(
            (a for k, a in _MOCK_ANSWERS if k in str(contents).lower()), _MOCK_DEFAULT)
        return _mock_response([types.Part(text="[MOCK] " + text)])
    log = []
    while fc:                                    # automatic mode: run the tool ourselves, like the SDK would
        fn = next(t for t in tools if callable(t) and t.__name__ == fc.name)
        out = fn(**dict(fc.args))
        log.append(f"{fc.name}({dict(fc.args)}) = {out}")
        items.append(types.Content(role="user", parts=[types.Part.from_function_response(name=fc.name, response={"result": out})]))
        fc = _mock_next_call(items, tools)
    text = ("[MOCK] tool calls: " + "; ".join(log) + ". (A live model would now phrase these results as an answer.)") if log \
        else "[MOCK] " + next((a for k, a in _MOCK_ANSWERS if k in str(contents).lower()), _MOCK_DEFAULT)
    return _mock_response([types.Part(text=text)])

def generate(**kw):
    """client.models.generate_content + retry on 429 (+ the MOCK stand-in). Same keyword arguments."""
    if MOCK:
        return mock_generate(**kw)
    for attempt in range(4):
        try:
            return client.models.generate_content(**kw)
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited..."); time.sleep(20*(attempt+1))
            else: raise

def ask(prompt, tools=None, temperature=None):
    """Plain-text answer. Pass tools=[fn, ...] and the SDK runs the calls for you (automatic function calling)."""
    r = generate(model=MODEL, contents=prompt,
                 config=types.GenerateContentConfig(tools=tools, temperature=temperature))
    return r.text or "[empty/blocked response]"
print("ready ✓" + ("  (MOCK mode — no API calls; tool calls are simulated)" if MOCK else ""))

## Part A — The calculator tool

The model never runs your function — it *asks* to, your code executes, the result goes back. The SDK reads your **docstring + type hints** to build the tool declaration, so write them like instructions.

The calculator below uses a small AST allow-list rather than Python `eval`. This is still a teaching tool: production tools also need input limits, timeouts, authorization, logging, and tests.

In [ ]:
# Cell 2 — define a tool as a plain Python function
import ast
import operator

_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _safe_math(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_safe_math(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _BIN_OPS:
        left, right = _safe_math(node.left), _safe_math(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("exponent too large")
        return _BIN_OPS[type(node.op)](left, right)
    raise ValueError("only numeric arithmetic is allowed")

def calculator(expression: str) -> float:
    """Evaluate a numeric arithmetic expression and return the exact result.
    Use this for ANY arithmetic. Never compute numbers yourself.
    Example expression: '2347 * 0.18'."""
    tree = ast.parse(expression, mode="eval")
    result = _safe_math(tree.body)
    if abs(result) > 1e12:
        raise ValueError("result too large")
    return float(result)

# Hand the model the tool; SDK does automatic function calling.
config = types.GenerateContentConfig(tools=[calculator])

r = generate(                       # = client.models.generate_content, with a 429 retry (Cell 1)
    model=MODEL,
    contents="What is 18% GST on a bill of 2347 rupees, and the total?",
    config=config)
print("WITH tool   :", r.text)

# the same question with no tool — watch it predict digits instead of computing them
print("WITHOUT tool:", ask("Without using any tool, what is 18% GST on a bill of 2347 rupees, and the total?"))

Ask the same thing **without** `config` (no tool) and compare — watch it guess digits. That contrast is the whole point.

### ✓ Checkpoint 1 — with-tool answer is exact; you saw the no-tool version wobble.

---
## Part B — A second tool + chaining

In [ ]:
# Cell 3 — add a tool; ask something needing BOTH
def days_between(date1: str, date2: str) -> int:
    """Return the number of days between two dates in YYYY-MM-DD format."""
    from datetime import date
    a = date.fromisoformat(date1); b = date.fromisoformat(date2)
    return abs((b - a).days)

config = types.GenerateContentConfig(tools=[calculator, days_between])

r = generate(
    model=MODEL,
    contents=("My internship is from 2026-05-15 to 2026-07-20 and pays 25000 per month. "
              "How many days is it, and roughly how much total if a month is 30 days?"),
    config=config)
print(r.text)

### ✓ Checkpoint 2 — it called BOTH tools to answer.

---
## Part C — See the machinery

Turn OFF automatic calling to watch every function call the model requests.

In [ ]:
# Cell 4 — manual loop: see the raw function calls
config = types.GenerateContentConfig(
    tools=[calculator, days_between],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

r = generate(
    model=MODEL, contents="What is 15% of 8400 plus 200?", config=config)

for part in r.candidates[0].content.parts:
    if part.function_call:
        print("MODEL WANTS:", part.function_call.name, dict(part.function_call.args))
    elif part.text:
        print("MODEL SAYS:", part.text)
# This is the request the model emits — YOUR code decides whether to run it.

## Part D — Scenario cards (paper + pen, no code)

For each, pick **Prompt / RAG / Tools / Fine-tune** and justify in 2 sentences. This is a mini design doc — and capstone rehearsal.

1. A bot that answers questions about your college's 80-page attendance & exam rulebook, with citations.
2. Replies are correct but too long and too formal; you want short and friendly.
3. "What's the weather in Madurai right now, and should I carry an umbrella to the exam?"
4. A model that must emit your exact 10-field JSON ticket format 50,000×/day on a small cheap model.

### ✓ Checkpoint 3 — defend one choice to me out loud.

---
## Stretch goals

## Reload your Lab 4 store (for Stretch 1 and Session 6)

Lab 4 Cell 4 saved `chunk_vecs.npy` + `chunks.json` to your Drive (`MyDrive/genai`). This cell mounts Drive, reloads them and re-defines `embed()` / `search()` — no re-embedding, no quota. If nothing is found it falls back to three demo chunks so the cells below still run; your capstone wants **your** notes, so re-run Lab 4 Cell 4 if you see that notice.

In [ ]:
# Reload — your Lab 4 vector store, back from Drive
import os, json, re, zlib
import numpy as np

EMBED_MODEL = "gemini-embedding-2"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/genai'
except Exception as e:
    print("Drive not mounted (" + type(e).__name__ + ") — looking on the local runtime disk instead")
    SAVE_DIR = '.'

def embed(texts):
    """One 768-d vector per text.
    Why the wrapping: gemini-embedding-2 folds a bare list of strings into ONE aggregated embedding
    (60 chunks → 1 vector, silently). Wrapping each text as its own Content gives one vector per chunk."""
    if MOCK:   # offline: deterministic hashed bag-of-words (crc32, so it matches across sessions) — search still ranks by word overlap
        M = np.zeros((len(texts), 768))
        for i, t in enumerate(texts):
            for w in re.findall(r"[a-z0-9]{4,}", t.lower()):      # skip "the", "of", "is"…
                M[i, zlib.crc32(w.encode()) % 768] += 1
        return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[types.Content(parts=[types.Part.from_text(text=t)]) for t in texts],
        config=types.EmbedContentConfig(output_dimensionality=768))
    assert len(res.embeddings) == len(texts), \
        f"expected {len(texts)} vectors, got {len(res.embeddings)} — each text must be wrapped as its own Content"
    return np.array([e.values for e in res.embeddings])

def search(query, k=3):
    qv = embed([query])[0]
    qv = qv / np.linalg.norm(qv)
    scores = chunk_vecs @ qv                  # cosine similarity, all chunks at once
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in top]

try:
    chunk_vecs = np.load(os.path.join(SAVE_DIR, "chunk_vecs.npy"))
    chunks = json.load(open(os.path.join(SAVE_DIR, "chunks.json"), encoding="utf-8"))
    print("reloaded", chunk_vecs.shape, "vectors +", len(chunks), "chunks from", SAVE_DIR)
except FileNotFoundError:
    print("no saved store in", SAVE_DIR, "— using 3 demo chunks. Re-run Lab 4 Cell 4 to get YOUR notes back.")
    chunks = ["Attendance: a minimum of 75% attendance is required to write the end-semester exam; 65-75% may be condoned on medical grounds.",
              "Passing a theory course needs 45% in the end-semester exam and 50% of the total marks (internal + end-semester).",
              "Internal assessment: two tests of 20 marks scaled to 30, plus 20 marks for assignments and quizzes."]
    chunk_vecs = embed(chunks)
    chunk_vecs = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
s, c = search("minimum marks needed to pass the end-semester examination")[0]
print(f"search() ready — top hit for 'pass mark': {s:.2f} | {c[:70]}...")

In [ ]:
# Stretch 1 — plug your Session 4 RAG in as a TOOL (capstone move!)
# search() and chunk_vecs come from the reload cell above (your Lab 4 store, back from Drive).

def search_notes(query: str) -> str:
    """Search the student's personal course notes and return the most relevant passages.
    Use this for any question about the student's specific courses, syllabus, or college."""
    hits = search(query, k=3)      # from the reload cell (Lab 4's search)
    return "\n\n".join(c for s, c in hits)

config = types.GenerateContentConfig(tools=[calculator, search_notes])
r = generate(
    model=MODEL,
    contents="According to my notes, what's the pass mark — and if I have 12/25 internal, what % of the end-sem do I need?",
    config=config)
print(r.text)
# Your capstone is now: knowledge (RAG) + hands (tools). One assistant.

In [ ]:
# Stretch 2 — break it, then fix it
# Give the model a badly-described tool and watch it misfire:
def mystery(x: str) -> str:
    """Does stuff."""        # <- terrible docstring on purpose
    return "42"
# Ask something ambiguous with [calculator, mystery] as tools.
# Then rewrite the docstring to be specific and watch behaviour change.
# Lesson: the docstring IS the prompt.

### S3 · Grade the trajectory, not just the answer

Session 2 taught you to score answers. An agent needs its *path* scored too — a right answer reached by a wrong route is a bug you haven't found yet. Tool-choice accuracy is the cheapest useful agent metric: no full run, five questions, and it moves the moment a docstring gets vague. Note the `None` case — calling a tool you didn't need is as much a failure as missing one.

### S4 · Put the loop on a leash

A leash = a max-steps cap plus a human gate on risky tools. Run it, then watch what the cap catches — the run that hits `MAX_STEPS` is the one that would have wandered on your bill.



In [ ]:
# Stretch 3 — grade the TRAJECTORY, not just the answer (Session 2, now for agents)
# An agent can reach a RIGHT answer by a stupid, expensive or dangerous path — and a
# score that only reads the final text will never tell you. So measure the FIRST decision:
# given a question, does it reach for the correct tool? This needs no full run, and it is
# the fastest signal that a docstring needs rewriting.
tool_tests = [
    ("What is 15% of 8400?",                       "calculator"),
    ("How many days from 2026-01-01 to 2026-03-01?", "days_between"),
    ("Who wrote the Thirukkural?",                  None),   # ← no tool needed. Over-calling is a bug too.
    ("What is 12 * 37 plus 8?",                     "calculator"),
    ("Days between 2026-05-15 and 2026-07-20?",     "days_between"),
]

probe = types.GenerateContentConfig(
    tools=[calculator, days_between],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

def first_tool(question):
    """Return the name of the tool the model reaches for first, or None if it answers directly."""
    r = generate(model=MODEL, contents=question, config=probe)
    for part in r.candidates[0].content.parts:
        if part.function_call:
            return part.function_call.name
    return None

right = 0
for q, expected in tool_tests:
    got = first_tool(q)
    ok = (got == expected); right += ok
    print(f"{'✓' if ok else '✗'} {str(got):14s} (wanted {str(expected):14s})  {q[:44]}")
    time.sleep(2)
print(f"\ntool-choice accuracy: {right}/{len(tool_tests)}")
# Now go and WEAKEN one docstring, re-run, and watch this number drop. That is the
# fastest signal a docstring needs rewriting.


In [ ]:
# Stretch 4 — the leash: what the demo loop is missing
# The naive agent loop is `while response.wants_tool_call: run it, send the result back` — its only
# exit condition is the model deciding it is done: an unbounded loop whose termination is decided by a
# probabilistic system, holding your key. The guarded loop below adds the leash: a step cap and a
# tool allow-list (the SDK's automatic calling in Cells 2–3 has its own internal cap, too).
MAX_STEPS = 5
ALLOWED   = {"calculator", "days_between"}

def guarded_run(question, max_steps=MAX_STEPS):
    contents = [question]
    calls, steps = [], 0
    while steps < max_steps:
        r = generate(model=MODEL, contents=contents, config=probe)
        parts = r.candidates[0].content.parts
        fc = next((p.function_call for p in parts if p.function_call), None)
        if not fc:
            return next((p.text for p in parts if p.text), ""), calls, steps
        if fc.name not in ALLOWED:                    # it CAN invent tool names
            return f"[blocked: unknown tool {fc.name}]", calls, steps
        args = dict(fc.args)
        try:
            out = {"calculator": calculator, "days_between": days_between}[fc.name](**args)
        except Exception as e:
            out = f"error: {e}"                       # readable errors -> models self-correct
        calls.append((fc.name, args, out))
        contents += [r.candidates[0].content,
                     types.Content(role="user", parts=[types.Part.from_function_response(
                         name=fc.name, response={"result": out})])]
        steps += 1
    return "[stopped: hit MAX_STEPS]", calls, steps

answer, calls, steps = guarded_run(
    "My internship runs 2026-05-15 to 2026-07-20 at 25000 per month. Days, and total at 30 days/month?")
print(answer, "\n")
for name, args, out in calls: print(f"  called {name}{args} -> {out}")
print(f"\nsteps used: {steps}/{MAX_STEPS}   <- log this. A task that usually takes 2 and sometimes takes 9 is telling you it got lost — log the distribution, not the mean.")


## Capstone: knowledge + hands + judge

You now have RAG (S4) + tools (S5) + evals (S2). **Save the notebook.** Final session: we attack it, harden it, and you demo. Last break — then the finale.